# Week 58, Day 3: Bagging and random forests

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w058/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(583)
P = 12

def trials(n, g):
    shared = g.normal(size=(n, 1))
    sig = 0.7 * shared + g.normal(size=(n, 4))                    # four correlated choice-related neurons
    z = 1.5 * sig[:, 0] + 1.0 * sig[:, 1] - 1.0 * sig[:, 2] * sig[:, 3]   # with a nonlinear interaction
    y = (g.random(n) < 1 / (1 + np.exp(-2 * z))).astype(int)
    return np.c_[sig, g.normal(size=(n, P - 4))], y

def gini_counts(c):
    n = c.sum(-1, keepdims=True); return 1 - np.sum((c / np.maximum(n, 1)) ** 2, axis=-1)

def best_split(X, y, feats):
    best = (None, None, np.inf); n = len(y)
    for j in feats:
        o = np.argsort(X[:, j]); xs, ys = X[o, j], y[o]
        left = np.cumsum(np.eye(2)[ys], axis=0)[:-1]; right = left[-1] + np.eye(2)[ys[-1]] - left
        nl = np.arange(1, n); imp = (nl * gini_counts(left) + (n - nl) * gini_counts(right)) / n
        imp = np.where(xs[1:] > xs[:-1], imp, np.inf); i = int(np.argmin(imp))
        if imp[i] < best[2]: best = (j, (xs[i] + xs[i + 1]) / 2, imp[i])
    return best

def grow(X, y, g, mtry=None, min_leaf=2):
    counts = np.bincount(y, minlength=2)
    if len(y) < 2 * min_leaf or counts.max() == len(y): return ("leaf", counts[1] / counts.sum())
    feats = range(X.shape[1]) if mtry is None else g.choice(X.shape[1], mtry, replace=False)
    j, t, imp = best_split(X, y, feats)
    if j is None: return ("leaf", counts[1] / counts.sum())
    m = X[:, j] <= t
    if m.all() or (~m).all(): return ("leaf", counts[1] / counts.sum())
    return ("node", j, t, grow(X[m], y[m], g, mtry, min_leaf), grow(X[~m], y[~m], g, mtry, min_leaf))

def predict(tree, X):
    if tree[0] == "leaf": return np.full(len(X), tree[1])
    _, j, t, L, R = tree; m = X[:, j] <= t; out = np.empty(len(X))
    out[m], out[~m] = predict(L, X[m]), predict(R, X[~m]); return out

X, y = trials(250, r); X_te, y_te = trials(2000, r)
acc = lambda p, y_: np.mean((p > 0.5) == (y_ == 1))
print("ready: 250 training trials of 12 neurons, 2,000 test trials")

### Exercise: An average of deep trees

Grow 30 deep trees, each on a bootstrap resample, using all features at every split. Average their predicted probabilities and record test accuracy for ensembles of 1, 3, 10 and 30 trees.

In [ ]:
g = np.random.default_rng(1)
bag_preds = []
for b in range(30):
    idx = g.integers(0, 250, 250)
    tree = grow(X[idx], y[idx], g)
    bag_preds.append(...)
bag_preds = np.array(bag_preds)
sizes = [1, 3, 10, 30]
bag_acc = [acc(bag_preds[:k].mean(0), y_te) for k in sizes]
for k, a in zip(sizes, bag_acc):
    print(f"{k:2d} bagged trees: test accuracy {a:.3f}")

In [ ]:
import numpy as np
def test_curve():
    "bag_acc has one accuracy per ensemble size, and 30 trees beat one"
    assert len(bag_acc) == len(sizes) and bag_acc[-1] > bag_acc[0] + 0.03

test_curve()
print("All checks passed")

### Exercise: Decorrelating the trees

Grow 30 trees as before, but with mtry = 6 features considered at each split. Compute the mean pairwise correlation between trees' predicted probabilities on the test set, for the bagged trees and the forest, and the forest's test accuracy.

In [ ]:
rf_preds = []
for b in range(30):
    idx = g.integers(0, 250, 250)
    rf_preds.append(predict(grow(X[idx], y[idx], g, mtry=6), X_te))
rf_preds = np.array(rf_preds)

def mean_corr(P):
    C = np.corrcoef(P)
    return ...

rho_bag, rho_rf = mean_corr(bag_preds), mean_corr(rf_preds)
acc_rf = acc(rf_preds.mean(0), y_te)
print(f"mean correlation between trees: bagging {rho_bag:.3f}, random forest {rho_rf:.3f}")
print(f"test accuracy with 30 trees: bagging {bag_acc[-1]:.3f}, random forest {acc_rf:.3f}")

In [ ]:
import numpy as np
def test_corr():
    "trees in the random forest are less correlated than bagged trees"
    assert rho_rf < rho_bag
def test_acc():
    "the 30-tree random forest is at least as accurate as 30 bagged trees"
    assert acc_rf >= bag_acc[-1] - 0.005

test_corr()
test_acc()
print("All checks passed")

### Exercise: A free estimate of test accuracy

Grow a 30-tree random forest, remembering each tree's bootstrap indices. For each training trial, average the predictions of the trees for which it was out of bag, and compute the out-of-bag accuracy. Compare it with the forest's test accuracy.

In [ ]:
votes, n_votes, oob_frac, te_preds = np.zeros(250), np.zeros(250), [], []
for b in range(30):
    idx = g.integers(0, 250, 250)
    out = np.setdiff1d(np.arange(250), idx)
    tree = grow(X[idx], y[idx], g, mtry=6)
    votes[out] += ...
    n_votes[out] += 1
    oob_frac.append(len(out) / 250); te_preds.append(predict(tree, X_te))
has = n_votes > 0
oob_acc = acc(votes[has] / n_votes[has], y[has])
test_acc_forest = acc(np.mean(te_preds, 0), y_te)
print(f"each tree left out {100 * np.mean(oob_frac):.1f}% of trials on average")
print(f"out-of-bag accuracy {oob_acc:.3f}; test accuracy {test_acc_forest:.3f}")

In [ ]:
import numpy as np
def test_frac():
    "each tree leaves out between 30% and 45% of the training trials"
    assert 0.3 < np.mean(oob_frac) < 0.45
def test_oob():
    "the out-of-bag accuracy is within 0.05 of the test accuracy"
    assert abs(oob_acc - test_acc_forest) < 0.05

test_frac()
test_oob()
print("All checks passed")